In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from qiskit.circuit.library import ZZFeatureMap, TwoLocal, EfficientSU2, RealAmplitudes, NLocal
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.circuit import QuantumCircuit, ParameterVector

def nlocal_feature_map(n, reps=2):
    qc = QuantumCircuit(n)
    params = ParameterVector('x', n)  # only n params — reused across reps
    for _ in range(reps):
        for q in range(n):
            qc.ry(params[q], q)  # reuse the same n params each rep
        for q in range(n - 1):
            qc.cx(q, q + 1)
    # final rotation layer
    for q in range(n):
        qc.ry(params[q], q)
    return qc

iris = load_iris()
X, y = iris.data, iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = {}

for n in [4, 2]:
    if n == 2:
        X_reduced = PCA(n_components=2).fit_transform(X_scaled)
    else:
        X_reduced = X_scaled

    X_train, X_test, y_train, y_test = train_test_split(
        X_reduced, y, test_size=0.2, random_state=42
    )

    # Classical SVM
    clf = SVC(kernel='rbf')
    clf.fit(X_train, y_train)
    results[f'classical_n{n}'] = (clf.score(X_train, y_train), clf.score(X_test, y_test))

    # Quantum SVMs
    feature_maps = {
        'ZZFeatureMap':   ZZFeatureMap(feature_dimension=n, reps=2),
        'TwoLocal':       TwoLocal(n, ['ry','rz'], 'cx', reps=2),
        'NLocal':         nlocal_feature_map(n, reps=2),
        'EfficientSU2':   EfficientSU2(n, reps=2),
        'RealAmplitudes': RealAmplitudes(n, reps=2),
    }

    for name, fm in feature_maps.items():
        kernel = FidelityQuantumKernel(feature_map=fm)
        qsvc = QSVC(quantum_kernel=kernel)
        qsvc.fit(X_train, y_train)
        results[f'{name}_n{n}'] = (qsvc.score(X_train, y_train), qsvc.score(X_test, y_test))

# Print table
print(f"{'Model':<20} {'Train(n=4)':<12} {'Train(n=2)':<12} {'Test(n=4)':<12} {'Test(n=2)'}")
models = ['classical', 'ZZFeatureMap', 'TwoLocal', 'NLocal', 'EfficientSU2', 'RealAmplitudes']
for m in models:
    tr4, te4 = results.get(f'{m}_n4', ('—','—'))
    tr2, te2 = results.get(f'{m}_n2', ('—','—'))
    print(f"{m:<20} {tr4:<12.3f} {tr2:<12.3f} {te4:<12.3f} {te2:.3f}")

C:\Users\dvite\AppData\Local\Temp\ipykernel_12852\420504285.py:52: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  'ZZFeatureMap':   ZZFeatureMap(feature_dimension=n, reps=2),
C:\Users\dvite\AppData\Local\Temp\ipykernel_12852\420504285.py:53: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  'TwoLocal':       TwoLocal(n, ['ry','rz'], 'cx', reps=2),
C:\Users\dvite\AppData\Local\Temp\ipykernel_12852\420504285.py:55: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Us

ValueError: Length of ('x[0]', 'x[1]', 'x[2]', 'x[3]', 'x[4]', 'x[5]', 'x[6]', 'x[7]', 'x[8]', 'x[9]', 'x[10]', 'x[11]', 'x[12]', 'x[13]', 'x[14]', 'x[15]', 'x[16]', 'x[17]', 'x[18]', 'x[19]', 'x[20]', 'x[21]', 'x[22]', 'x[23]', 'y[0]', 'y[1]', 'y[2]', 'y[3]', 'y[4]', 'y[5]', 'y[6]', 'y[7]', 'y[8]', 'y[9]', 'y[10]', 'y[11]', 'y[12]', 'y[13]', 'y[14]', 'y[15]', 'y[16]', 'y[17]', 'y[18]', 'y[19]', 'y[20]', 'y[21]', 'y[22]', 'y[23]') inconsistent with last dimension of [-1.50652052  1.24920112 -1.56757623 -1.3154443  -0.17367395  3.09077525
 -1.2833891  -1.05217993]